<a href="https://colab.research.google.com/github/CaoTrongNghia/dipoleMoment-dftEstimator/blob/main/dft_spice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rdkit
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install torch_geometric

# Clone TorchMD-NET fresh
!git clone https://github.com/torchmd/torchmd-net.git
%cd torchmd-net

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 60.0 MB/s eta 0:00:00
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.9 MB/s eta 0:00:00
Cloning into 'torchmd-net'...
remote: Enumerating objects: 8775, done.
remote: Counting objects: 100% (652/652), done.
remote: Compressing objects: 100% (226/226), done.
remote: Total 8775 (delta 518), reused 514 (delta 424), pack-reused 8123 (from 3)
Receiving objects: 100% (8775/8775), 189.03 MiB | 17.10 MiB/s, done.
Resolving deltas: 100% (6115/6115), done.
/content/torchmd-net


In [ ]:
# Install TorchMD-NET
!pip install -e . --no-cache-dir --default-timeout=200

# torchvision and torchaudio are not needed

Obtaining file:///content/torchmd-net
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 155.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 269.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 MB 303.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 354.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 402.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 354.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 344.6 MB/s eta 0:00:00
  Building editable for torchmd-net (pyproject.toml) ... done
  Created wheel for torchmd-net: filename=torchmd_net-3.0.3-0.editable-py3-none-any.whl size=9656 sha256=57592b4e9ac77afe99d5ff9b0a

In [ ]:
!pip install h5py ase
!pip install wandb

In [ ]:
import sys
sys.path.append("/usr/local/lib/python3.11/site-packages/")
sys.path.append("/content/torchmd-net")

In [ ]:
import urllib.request
import os

# Download SPICE 1.1.4 from Zenodo
url = "https://zenodo.org/records/8222043/files/SPICE-1.1.4.hdf5"
filename = "SPICE.hdf5"

if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)
    print("Download complete!")
else:
    print("SPICE dataset already exists.")

Download complete!


In [ ]:
import h5py
import torch
from torch.utils.data import Dataset
import numpy as np

class SPICEDataset(Dataset):
    def __init__(self, hdf5_path):
        super().__init__()
        self.hdf5_path = hdf5_path
        self.data = []

        print("Parsing HDF5 file...")
        with h5py.File(hdf5_path, 'r') as f:
            # SPICE groups data by subset, then by molecule ID
            for subset in f.keys():
                for mol_id in f[subset].keys():
                    mol_group = f[subset][mol_id]

                    # Extract static molecule properties
                    z = torch.tensor(mol_group['atomic_numbers'][:], dtype=torch.long)

                    # Extract conformation-specific properties
                    pos = mol_group['conformations'][:]          # Shape: (num_confs, num_atoms, 3)
                    energy = mol_group['dft_total_energy'][:]    # Shape: (num_confs,)
                    force = mol_group['dft_total_gradient'][:]   # Shape: (num_confs, num_atoms, 3)

                    # Store each conformation as a separate data point
                    for i in range(len(energy)):
                        self.data.append({
                            'z': z,
                            'pos': torch.tensor(pos[i], dtype=torch.float32),
                            'y': torch.tensor(energy[i], dtype=torch.float32),
                            'dy': torch.tensor(-force[i], dtype=torch.float32) # TorchMD-Net expects forces, gradient is negative force
                        })
        print(f"Loaded {len(self.data)} total conformations!")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# Instantiate it
# dataset = SPICEDataset("SPICE.hdf5")